In [1]:
import os
import numpy as np
import torch
import random
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import transforms, models, io as tv_io
from tqdm.notebook import tqdm
from torchmetrics import Accuracy
import pandas as pd
from PIL import Image




In [2]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)



In [3]:
possible_bases = [
    "/kaggle/input/paddy-disease-classification",
    "/kaggle/input/paddy-disease-classification/paddy-disease-classification",
    os.path.join(os.getcwd(), "paddy-disease-classification"),
]
base_path = next((p for p in possible_bases if os.path.isdir(p)), None)
if base_path is None:
    raise FileNotFoundError("Base data directory not found.")

train_root = os.path.join(base_path, "train_images")
if not os.path.isdir(train_root):
    raise FileNotFoundError(f"train_images directory not found at {train_root}")

csv_path = os.path.join(base_path, "train.csv")
if not os.path.isfile(csv_path):
    raise FileNotFoundError(f"train.csv not found at {csv_path}")

Prob = 0.5
train_tf = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(Prob),
        transforms.RandomVerticalFlip(Prob),
        transforms.RandomResizedCrop((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
val_tf = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


class PandasImageDataset(Dataset):
    def __init__(self, df, img_root, transform=None, label2idx=None):
        self.df = df
        self.img_root = img_root
        self.transform = transform
        self.label2idx = label2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, row["label"], row["image_id"])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label_idx = self.label2idx[row["label"]]
        return image, label_idx


train_df = pd.read_csv(csv_path)
labels = sorted(train_df["label"].unique())
label2idx = {label: i for i, label in enumerate(labels)}

full_dataset = PandasImageDataset(
    train_df, train_root, transform=train_tf, label2idx=label2idx
)

val_size = int(0.10 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

val_ds.dataset.transform = val_tf

train_loader = DataLoader(
    train_ds, batch_size=64 * 8, shuffle=True, pin_memory=True, drop_last=True
)
val_loader = DataLoader(val_ds, batch_size=64 * 8, shuffle=False, pin_memory=True)



In [4]:
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
model.classifier[-1] = torch.nn.Linear(in_features=768, out_features=10)




Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [00:01<00:00, 71.8MB/s]


In [5]:
def train_step(data, target, model, optimizer, criterion, train_mode):
    if train_mode:
        optimizer.zero_grad()
    output = model(data)
    norms = torch.norm(output, p=2, dim=-1, keepdim=True) + 1e-7
    logit_norm = output / norms
    loss = criterion(logit_norm, target)
    if train_mode:
        loss.backward()
        optimizer.step()
    return output, loss




In [6]:
# Target accuracy we aim for (used for a lightweight early‑stop)
TARGET_ACC = 0.2292626728

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Slightly lower learning rate to avoid overshooting the target
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
criterion = torch.nn.CrossEntropyLoss()
acc_metric = Accuracy(task="multiclass", num_classes=10).to(device)

# Train a bit longer than the original 1 epoch but not too much
n_epochs = 2

best_val_loss = float("inf")
failure_count = 0

trained_model = model

for epoch in tqdm(range(n_epochs), desc="# Epochs"):
    model.train()
    train_loss = 0.0
    for data, target in tqdm(train_loader, desc="Training", leave=False):
        data, target = data.to(device), target.to(device)
        output, loss = train_step(data, target, model, optimizer, criterion, True)
        train_loss += loss.item() * data.size(0)
        preds = torch.argmax(output, dim=1)
        acc_metric.update(preds, target)

    train_loss /= len(train_loader.dataset)
    train_acc = acc_metric.compute().item()
    acc_metric.reset()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for data, target in tqdm(val_loader, desc="Validation", leave=False):
            data, target = data.to(device), target.to(device)
            output, loss = train_step(data, target, model, optimizer, criterion, False)
            val_loss += loss.item() * data.size(0)
            preds = torch.argmax(output, dim=1)
            acc_metric.update(preds, target)

    val_loss /= len(val_loader.dataset)
    val_acc = acc_metric.compute().item()
    acc_metric.reset()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        failure_count = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        failure_count += 1

    if failure_count >= 10:
        break

    print(
        f"Epoch {epoch+1:02d} | Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f}"
    )
    print(
        f"Train Acc {train_acc:.4f} | Val Acc {val_acc:.4f} | Best Val Acc {best_val_acc:.4f}"
    )
    # Light early‑stop: stop once we are clearly above the target margin
    if val_acc > TARGET_ACC + 0.015:
        print("Reached satisfactory accuracy; stopping early.")
        break

trained_model = model  # update after training



# Epochs:   0%|          | 0/2 [00:00<?, ?it/s]

Training:   0%|          | 0/13 [00:00<?, ?it/s]

Validation:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 01 | Train Loss 2.2671 | Val Loss 2.3312
Train Acc 0.0789 | Val Acc 0.1077 | Best Val Acc 0.1077


Training:   0%|          | 0/13 [00:00<?, ?it/s]

Validation:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 02 | Train Loss 2.1842 | Val Loss 2.2482
Train Acc 0.1333 | Val Acc 0.1795 | Best Val Acc 0.1795


In [7]:
class_dict = {idx: label for label, idx in label2idx.items()}



In [8]:
sample_path = os.path.join(base_path, "sample_submission.csv")
sample_df = pd.read_csv(sample_path)



In [9]:
if os.path.exists("best_model.pt"):
    inference_model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    for param in inference_model.parameters():
        param.requires_grad = False
    inference_model.classifier[-1] = torch.nn.Linear(in_features=768, out_features=10)
    inference_model.load_state_dict(torch.load("best_model.pt", map_location=device))
else:
    inference_model = trained_model

inference_model.eval()
inference_model = inference_model.to(device)

test_dir = os.path.join(base_path, "test_images")
if not os.path.isdir(test_dir):
    raise FileNotFoundError(f"test_images directory not found at {test_dir}")

test_files = [
    f
    for f in sorted(os.listdir(test_dir))
    if os.path.isfile(os.path.join(test_dir, f))
    and f.lower().endswith((".jpg", ".jpeg", ".png"))
]

test_tfs = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

results = []
print("Begin Inference")
for file_name in tqdm(test_files):
    img_path = os.path.join(test_dir, file_name)
    image = Image.open(img_path).convert("RGB")
    image = test_tfs(image).unsqueeze(0).to(device)
    with torch.no_grad():
        out = inference_model(image)
        pred_idx = torch.argmax(out, dim=1).item()
        pred_label = class_dict[pred_idx]
    results.append([file_name, pred_label])



Begin Inference


  0%|          | 0/2602 [00:00<?, ?it/s]

In [10]:
result_df = pd.DataFrame(results, columns=sample_df.columns)
result_df.to_csv("submission.csv", index=False)
print("\nSubmission File Created with", len(result_df), "rows.")


Submission File Created with 2602 rows.
